In [2]:
#source URL:
# https://huggingface.co/datasets/stanfordnlp/sentiment140/blob/main/sentiment140.py
import os
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, random_split
import torch

from base import BaseDataLoader, DatasetSubset

class Sent140Dataset(Dataset):
    def __init__(self, data_dir, train=True, iid=False, num_clients=None, seq_len=25):
        super().__init__()
        self.data_dir = data_dir
        self.train = train
        self.iid = iid
        self.seq_len = seq_len
        
        # Load embeddings first
        self.word_to_idx, self.embeddings = self._load_embeddings()
        self.vocab_size = len(self.word_to_idx)
        self.unk_idx = self.word_to_idx.get('<UNK>', 0)
        
        # Read and process data
        self.train_data, self.test_data = self._read_data()
        self.num_clients = num_clients if num_clients else len(set(self.train_data['user']))
        
        # Prepare data
        if self.train:
            self.data, self.labels, self.client_indices = self._prepare_training_data()
        else:
            self.data, self.labels = self._prepare_test_data()

    def _load_embeddings(self):
        """Load pre-processed embeddings from embs.json"""
        embs_path = os.path.join(self.data_dir, 'embs.json')
        with open(embs_path, 'r') as f:
            embs_data = json.load(f)
        
        # Create word to index mapping
        word_to_idx = {word: idx for idx, word in enumerate(embs_data['vocab'])}
        embeddings = np.array(embs_data['emba'])
        
        return word_to_idx, embeddings

    def _read_data(self):
        """Read CSV data and perform basic preprocessing"""
        data_path = os.path.join(self.data_dir, 'sentiment140', 'training.1600000.processed.noemoticon.csv')
        
        # Read CSV file
        full_data = pd.read_csv(data_path, encoding='latin-1', header=None,
                              names=['sentiment', 'id', 'date', 'query', 'user', 'text'])
        
        # Convert sentiment labels (0=negative, 4=positive) to binary (0=negative, 1=positive)
        full_data['sentiment'] = (full_data['sentiment'] // 4).astype(int)
        
        # Split data into train/test sets (90% train, 10% test)
        np.random.seed(42)  # for reproducibility
        mask = np.random.rand(len(full_data)) < 0.9
        train_data = full_data[mask]
        test_data = full_data[~mask]
        
        return train_data, test_data

    def _tokenize_and_pad(self, text):
        """Convert text to sequence of word indices and pad/truncate to seq_len"""
        # Basic tokenization (split on whitespace)
        tokens = str(text).lower().split()
        
        # Convert tokens to indices
        indices = [self.word_to_idx.get(token, self.unk_idx) for token in tokens]
        
        # Pad or truncate to seq_len
        if len(indices) < self.seq_len:
            indices = indices + [self.unk_idx] * (self.seq_len - len(indices))
        else:
            indices = indices[:self.seq_len]
            
        return indices

    def _prepare_training_data(self):
        """Prepare training data with IID distribution"""
        # Process all texts and labels
        all_data = [self._tokenize_and_pad(text) for text in self.train_data['text']]
        all_labels = self.train_data['sentiment'].tolist()
        
        # Shuffle data
        indices = list(range(len(all_data)))
        np.random.shuffle(indices)
        
        # Split data evenly among clients
        samples_per_client = len(indices) // self.num_clients
        remaining_samples = len(indices) % self.num_clients
        client_indices = {}
        
        current_idx = 0
        for i in range(self.num_clients):
            # Distribute remaining samples evenly
            extra_sample = 1 if i < remaining_samples else 0
            client_size = samples_per_client + extra_sample
            
            end_idx = current_idx + client_size
            client_indices[i] = set(range(current_idx, end_idx))
            current_idx = end_idx
        
        return [all_data[i] for i in indices], [all_labels[i] for i in indices], client_indices

    def _prepare_test_data(self):
        """Prepare test data"""
        all_data = [self._tokenize_and_pad(text) for text in self.test_data['text']]
        all_labels = self.test_data['sentiment'].tolist()
        return all_data, all_labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        """Get a single item from the dataset"""
        sequence = torch.tensor(self.data[index], dtype=torch.long)
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return sequence, label

    def get_vocab_size(self):
        """Return the vocabulary size"""
        return self.vocab_size

    def get_embeddings(self):
        """Return the embedding matrix"""
        return torch.FloatTensor(self.embeddings)

class Sent140DataLoader(BaseDataLoader):
    def __init__(self, data_dir: str, n_clients: int, batch_size: int, iid: bool = True):
        super().__init__(data_dir, n_clients, batch_size, iid)

    def load_data(self):
        # Load datasets
        train_dataset = Sent140Dataset(
            data_dir=self.data_dir,
            train=True,
            iid=self.iid,
            num_clients=self.n_clients
        )
        test_dataset = Sent140Dataset(
            data_dir=self.data_dir,
            train=False
        )

        # Split test into validation and test
        test_size = len(test_dataset)
        val_size = int(0.2 * test_size)
        test_size = test_size - val_size
        val_dataset, test_dataset = random_split(test_dataset, [val_size, test_size])

        # Create dataloaders
        train_loaders = [
            DataLoader(
                DatasetSubset(train_dataset, train_dataset.client_indices[i]),
                batch_size=self.batch_size,
                shuffle=True
            ) for i in range(self.n_clients)
        ]
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)
        
        # Return the embedding matrix as additional data
        embeddings = train_dataset.get_embeddings()

        return train_loaders, val_loader, test_loader, embeddings

In [5]:

data_dir = "../data/Sent140"
n_clients=10
batch_size=32


In [6]:
print("Initializing dataloader...")
dataloader = Sent140DataLoader(data_dir=data_dir, n_clients=n_clients, batch_size=batch_size)
train_loaders, val_loader, test_loader, embeddings = dataloader.load_data()

Initializing dataloader...


In [4]:
# 1. Basic Dataset Statistics
print("\n=== Basic Dataset Statistics ===")
total_train = sum(len(loader.dataset) for loader in train_loaders)
print(f"Total training samples: {total_train}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")


=== Basic Dataset Statistics ===
Total training samples: 1440201
Validation samples: 31959
Test samples: 127840


In [5]:
# 2. Client Distribution Statistics
print("\n=== Client Distribution Statistics ===")
client_sizes = [len(loader.dataset) for loader in train_loaders]
print(f"Number of clients: {n_clients}")
print(f"Average samples per client: {np.mean(client_sizes):.2f}")
print(f"Min samples per client: {min(client_sizes)}")
print(f"Max samples per client: {max(client_sizes)}")
print(f"Std of samples per client: {np.std(client_sizes):.2f}")


=== Client Distribution Statistics ===
Number of clients: 10
Average samples per client: 144020.10
Min samples per client: 144020
Max samples per client: 144021
Std of samples per client: 0.30


In [6]:

from collections import Counter
# 3. Vocabulary and Embedding Statistics
print("\n=== Vocabulary and Embedding Statistics ===")
print(f"Vocabulary size: {len(embeddings)}")
print(f"Embedding dimension: {embeddings.shape[1]}")

# 4. Label Distribution
print("\n=== Label Distribution ===")
train_labels = []
for loader in train_loaders:
    for _, labels in loader:
        train_labels.extend(labels.numpy())
label_dist = Counter(train_labels)
for label, count in sorted(label_dist.items()):
    print(f"Label {label}: {count} samples ({count/len(train_labels)*100:.2f}%)")
    


=== Vocabulary and Embedding Statistics ===
Vocabulary size: 92642
Embedding dimension: 300

=== Label Distribution ===
Label 0: 720146 samples (50.00%)
Label 1: 720055 samples (50.00%)


In [7]:
# 5. Sequence Length Statistics
print("\n=== Sequence Length Statistics ===")
seq_lengths = []
for loader in train_loaders:
    for sequences, _ in loader:
        seq_lengths.extend([torch.sum(seq != loader.dataset.dataset.unk_idx).item() 
                            for seq in sequences])

print(f"Average sequence length: {np.mean(seq_lengths):.2f}")
print(f"Min sequence length: {min(seq_lengths)}")
print(f"Max sequence length: {max(seq_lengths)}")
print(f"Median sequence length: {np.median(seq_lengths):.2f}")


=== Sequence Length Statistics ===
Average sequence length: 10.29
Min sequence length: 0
Max sequence length: 25
Median sequence length: 9.00


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Sent140LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=100, num_classes=2, seq_len=25, num_layers=2):
        super(Sent140LSTM, self).__init__()
        self.embed = nn.Embedding(vocab_size , embedding_dim)  # +1 for unknown token
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_dim, 128)
        self.out = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embed(x)  # Embed the input
        x, _ = self.lstm(x)  # Pass through LSTM
        x = self.fc1(x[:, -1, :])  # Use the last hidden state
        x = self.dropout(x)
        x = self.out(x)  # Final output
        return x

In [ ]:
# 6. Model Compatibility Check
print("\n=== Model Compatibility Check ===")
# Sample batch for testing
sample_batch, sample_labels = next(iter(train_loaders[0]))
print(f"Batch shape: {sample_batch.shape}")
print(f"Labels shape: {sample_labels.shape}")

# Initialize model with correct parameters
model = Sent140LSTM(
    vocab_size=len(embeddings),
    embedding_dim=embeddings.shape[1],
    seq_len=sample_batch.shape[1]
    )

In [10]:
# Test forward pass
try:
    output = model(sample_batch)
    print(f"Model output shape: {output.shape}")
    print("✓ Forward pass successful")
    print(f"✓ Output dimension matches number of classes: {output.shape[-1]} classes")
except Exception as e:
    print(f"✗ Forward pass failed: {str(e)}")

# 7. Memory Usage Estimation
print("\n=== Memory Usage Estimation ===")
batch_size = train_loaders[0].batch_size
sample_batch_size = sample_batch.element_size() * sample_batch.nelement()
sample_labels_size = sample_labels.element_size() * sample_labels.nelement()
print(f"Single batch memory usage: {(sample_batch_size + sample_labels_size) / 1024 / 1024:.2f} MB")
print(f"Embedding matrix memory usage: {embeddings.nelement() * 4 / 1024 / 1024:.2f} MB")


Model output shape: torch.Size([32, 2])
✓ Forward pass successful
✓ Output dimension matches number of classes: 2 classes

=== Memory Usage Estimation ===
Single batch memory usage: 0.01 MB
Embedding matrix memory usage: 106.02 MB


In [11]:
# Basic dataset statistics
print("\n=== Basic Dataset Statistics ===")
total_train = sum(len(loader.dataset) for loader in train_loaders)
print(f"Total training samples: {total_train}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")

# Per-client statistics and label distribution
print("\n=== Per-Client Statistics ===")
for client_id, loader in enumerate(train_loaders):
    client_labels = []
    for _, labels in loader:
        client_labels.extend(labels.numpy())
    
    label_dist = Counter(client_labels)
    n_samples = len(client_labels)
    
    print(f"\nClient {client_id}:")
    print(f"Total samples: {n_samples}")
    for label, count in sorted(label_dist.items()):
        print(f"Label {label}: {count} samples ({count/n_samples*100:.2f}%)")



=== Basic Dataset Statistics ===
Total training samples: 1440201
Validation samples: 31959
Test samples: 127840

=== Per-Client Statistics ===

Client 0:
Total samples: 144021
Label 0: 71833 samples (49.88%)
Label 1: 72188 samples (50.12%)

Client 1:
Total samples: 144020
Label 0: 72004 samples (50.00%)
Label 1: 72016 samples (50.00%)

Client 2:
Total samples: 144020
Label 0: 72200 samples (50.13%)
Label 1: 71820 samples (49.87%)

Client 3:
Total samples: 144020
Label 0: 71907 samples (49.93%)
Label 1: 72113 samples (50.07%)

Client 4:
Total samples: 144020
Label 0: 72066 samples (50.04%)
Label 1: 71954 samples (49.96%)

Client 5:
Total samples: 144020
Label 0: 72160 samples (50.10%)
Label 1: 71860 samples (49.90%)

Client 6:
Total samples: 144020
Label 0: 71790 samples (49.85%)
Label 1: 72230 samples (50.15%)

Client 7:
Total samples: 144020
Label 0: 72318 samples (50.21%)
Label 1: 71702 samples (49.79%)

Client 8:
Total samples: 144020
Label 0: 71884 samples (49.91%)
Label 1: 72136 

In [12]:
# Set device
device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda:3


In [13]:
n_clients = 1  # We'll use just one client for testing
batch_size = 32

dataloader = Sent140DataLoader(data_dir=data_dir, n_clients=n_clients, batch_size=batch_size)
train_loaders, val_loader, test_loader, embeddings = dataloader.load_data()

# Get the first (and only) client's dataloader
train_loader = train_loaders[0]

# Print dataset sizes
print("\n=== Dataset Sizes ===")
print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")


=== Dataset Sizes ===
Training samples: 1440201
Validation samples: 31959
Test samples: 127840


In [7]:
# Initialize model with pre-trained embeddings
model = Sent140LSTM(
    vocab_size=len(embeddings),
    embedding_dim=embeddings.shape[1],  # Should be 300 for GloVe
    hidden_dim=100,
    num_classes=2,
    seq_len=25,
    num_layers=2
)

# Initialize the embedding layer with pre-trained embeddings
model.embed.weight.data.copy_(torch.FloatTensor(embeddings))
model.embed.weight.requires_grad = True  # Set to False if you want to freeze embeddings


In [8]:
# print all model parameters
print("\n=== Model Parameters ===")
for name, param in model.named_parameters():
    print(f"{name}: {param.size()}")
    if param.requires_grad:
        print(f"Gradient: {param.requires_grad}")



=== Model Parameters ===
embed.weight: torch.Size([92642, 300])
Gradient: True
lstm.weight_ih_l0: torch.Size([400, 300])
Gradient: True
lstm.weight_hh_l0: torch.Size([400, 100])
Gradient: True
lstm.bias_ih_l0: torch.Size([400])
Gradient: True
lstm.bias_hh_l0: torch.Size([400])
Gradient: True
lstm.weight_ih_l1: torch.Size([400, 100])
Gradient: True
lstm.weight_hh_l1: torch.Size([400, 100])
Gradient: True
lstm.bias_ih_l1: torch.Size([400])
Gradient: True
lstm.bias_hh_l1: torch.Size([400])
Gradient: True
fc1.weight: torch.Size([128, 100])
Gradient: True
fc1.bias: torch.Size([128])
Gradient: True
out.weight: torch.Size([2, 128])
Gradient: True
out.bias: torch.Size([2])
Gradient: True


In [ ]:
from tqdm import tqdm
# Move model to device
model = model.to(device)

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc='Training')
    for batch_idx, (inputs, targets) in enumerate(progress_bar):
        # Move to device
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': total_loss/(batch_idx+1),
            'acc': 100.*correct/total
        })
    
    return total_loss/len(train_loader), correct/total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return total_loss/len(loader), correct/total

# Training settings
epochs = 5
lr = 0.001
weight_decay = 0

# Initialize optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()

# Training loop
print("\n=== Starting Training ===")
best_val_acc = 0

for epoch in range(epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    # Print epoch statistics
    print(f"\nEpoch {epoch+1}/{epochs}:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"New best validation accuracy: {best_val_acc*100:.2f}%")
        torch.save(model.state_dict(), 'best_sent140_model.pt')

# Test final model
print("\n=== Testing Best Model ===")
model.load_state_dict(torch.load('best_sent140_model.pt'))
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc*100:.2f}%")